
# Notebook 10 — DFT candidate selection and input generation

**Project:** CMT Path A — leakage-audited multi-ion computed insertion-electrode benchmark
**Notebook:** `10_dft_candidate_selection_and_input_generation.ipynb`

## Purpose

This notebook selects a small, reviewer-defensible set of sodium-ion candidates for **original first-principles / DFT spot-check validation** and prepares input folders for relaxation/static calculations.

It does **not** run DFT and it does **not** make discovery claims. The goal is to create a reproducible bridge from the sodium case-study triage to Notebook 15, where completed DFT results will be parsed and compared against Materials Project computed electrode records.

## Inputs expected

Main required inputs:

```text
the canonical Notebook 08 repository artifacts/processed/08_sodium_top30_candidates_for_literature_review.csv
the canonical Notebook 09 repository artifacts/processed/09_candidate_literature_analogue_summary.csv
the canonical Notebook 09 repository artifacts/processed/09_manual_literature_annotations_WITH_CARRYOVER.csv
```

Optional inputs from earlier notebooks are used if present.

## Outputs

All outputs are saved under:

```text
the canonical Notebook 10 repository artifacts/
```

Major outputs:

```text
processed/10_dft_candidate_selection_scores.csv
processed/10_dft_primary_selection.csv
processed/10_dft_optional_control_selection.csv
processed/10_dft_reserve_candidates.csv
processed/10_dft_structure_resolution_table.csv
processed/10_dft_input_generation_manifest.csv
metadata/10_final_decision.json
dft_inputs/<candidate folders>/...
```

## Reviewer-safety rules

1. Select only **3–5 primary candidates**.
2. Include at most one optional control candidate if useful.
3. Penalize literature-weak, high-uncertainty, out-of-domain, high-volume-change, and chemically risky candidates.
4. Do not include POTCAR files in the repository.
5. Generated VASP inputs are templates; verify DFT+U, pseudopotentials, magnetism, and convergence before production runs.


In [ ]:

# ============================================================
# Cell 1 — Imports, configuration, and paths
# ============================================================
from __future__ import annotations

import os
import re
import json
import math
import shutil
import platform
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# -
# User controls
# -
PRIMARY_DFT_CANDIDATE_COUNT = 5      # CMT-safe: 3 to 5 recommended
INCLUDE_OPTIONAL_CONTROL = True      # optional calibration/control structure, not counted as primary
MAX_OPTIONAL_CONTROLS = 1

# Set this to explicit IDs only if you need to override the automatic selector.
# Example: MANUAL_PRIMARY_CANDIDATE_IDS = ["Na_candidate_10", "Na_candidate_11", "Na_candidate_15"]
MANUAL_PRIMARY_CANDIDATE_IDS = []
MANUAL_CONTROL_CANDIDATE_IDS = []

# Reviewer-safe defaults for the current CMT Path A sodium case-study stage.
# These avoid the top-ranked but chemically risky fluoride/very-common oxide cases as primary DFT targets.
# They can be changed after inspecting the selection table.
USE_REVIEWER_SAFE_DEFAULT_PRIMARY_IDS = True
REVIEWER_SAFE_DEFAULT_PRIMARY_IDS = [
    "Na_candidate_06",  # Na3CoPO4CO3 carbonophosphate
    "Na_candidate_10",  # Na3FePO4CO3 carbonophosphate
    "Na_candidate_11",  # Na3MnPO4CO3 carbonophosphate
    "Na_candidate_15",  # Na3Cr2(PO4)3 NASICON-like phosphate
    "Na_candidate_16",  # Na2MnP2O7 pyrophosphate
]
REVIEWER_SAFE_DEFAULT_CONTROL_IDS = ["Na_candidate_02"]  # known NaCoO2 oxide control/calibration, optional

# Chemical/DFT-risk gates. These are deliberately conservative.
HARD_EXCLUDE_AS_CONTAINING = True
HARD_EXCLUDE_H_CONTAINING = False       # H-containing framework is not impossible, but often hydration/proton ambiguity exists
HARD_EXCLUDE_NO_RELEVANT_LITERATURE = True
HARD_EXCLUDE_OUT_OF_DOMAIN = True
HARD_EXCLUDE_HIGH_UNCERTAINTY = False
HARD_EXCLUDE_SIMPLE_HALIDE_FLUORIDE_PRIMARY = True  # excludes halide_or_fluoride_like primary targets; fluorophosphates remain allowed
MAX_VOLUME_CHANGE_FOR_PRIMARY = 0.20    # high-volume cases can still be reserve candidates
MAX_STABILITY_WORST_FOR_PRIMARY = 0.10  # eV/atom-like threshold if Notebook 08 used eV/atom stability feature

# Structure retrieval and input generation.
RUN_MP_STRUCTURE_RESOLUTION = True
GENERATE_VASP_INPUTS = True
GENERATE_QE_PLACEHOLDER_INPUTS = True

# Materials Project API API handling: never paste your key into the notebook or save it in outputs.
# Preferred: set environment variable MP_API_KEY before running the notebook.
# If MP_API_KEY is missing, a hidden getpass prompt is enabled.
ASK_FOR_MP_API_KEY_IF_MISSING = True    # secure hidden prompt if MP_API_KEY env var is missing

# VASP/Pymatgen input mode.
# The notebook tries MPRelaxSet first if pymatgen is installed. If not possible, it writes safe template files.
VASP_SET_MODE = "MPRelaxSet_if_available_else_template"

# -
# Paths
# -
BASE_DIR = Path("outputs") / "Notebook 10"
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
DFT_INPUT_DIR = BASE_DIR / "dft_inputs"
LOG_DIR = BASE_DIR / "logs"

for d in [BASE_DIR, PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, DFT_INPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
LOG_ROWS = []


def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str, ensure_ascii=False),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")


def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "10_event_log.csv", index=False)


def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)


def clean_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def safe_filename(s: str, max_len: int = 90) -> str:
    s = clean_str(s)
    s = s.replace("/", "_").replace("\\", "_").replace(" ", "")
    s = re.sub(r"[^A-Za-z0-9_.+()\-]+", "_", s)
    return s[:max_len].strip("_") or "unnamed"

print("Notebook 10 initialized")
print("Output directory:", BASE_DIR)
log_event("init", "INFO", "Notebook 10 initialized", {"base_dir": str(BASE_DIR)})
save_event_log()


In [ ]:

# ============================================================
# Cell 2 — Locate and load Notebook 08 and 13 outputs
# ============================================================

def first_existing(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None

TOP30_CANDIDATE_PATHS = [
    Path("data/processed/notebook_09/09_sorted_top30_candidates.csv"),
    Path("data/processed/notebook_08/08_sodium_top30_candidates_for_literature_review.csv"),
]
LIT_SUMMARY_PATHS = [
    Path("data/processed/notebook_09/09_candidate_literature_analogue_summary.csv"),
]
LIT_DETAIL_PATHS = [
    Path("data/processed/notebook_09/09_manual_literature_annotations_WITH_CARRYOVER.csv"),
    Path("provenance/supplementary/notebook_09/manual_inputs/09_manual_literature_annotations_FILLED.csv"),
]

TOP30_PATH = first_existing(TOP30_CANDIDATE_PATHS)
LIT_SUMMARY_PATH = first_existing(LIT_SUMMARY_PATHS)
LIT_DETAIL_PATH = first_existing(LIT_DETAIL_PATHS)

missing = []
if TOP30_PATH is None:
    missing.append("top-30 candidate table from Notebook 08/13")
if LIT_SUMMARY_PATH is None:
    missing.append("Notebook 09 candidate literature summary")
if LIT_DETAIL_PATH is None:
    missing.append("Notebook 09 detailed manual literature annotations")

if missing:
    raise FileNotFoundError(
        "Missing required Notebook 10 input(s): " + "; ".join(missing) + "\n"
        "Run Notebook 08 and Notebook 09 first, and ensure their outputs are under outputs/."
    )

candidates = pd.read_csv(TOP30_PATH, low_memory=False)
lit_summary = pd.read_csv(LIT_SUMMARY_PATH, low_memory=False)
lit_detail = pd.read_csv(LIT_DETAIL_PATH, low_memory=False)

# Basic validation.
required_candidate_cols = [
    "manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_charge",
    "formula_discharge", "framework_formula", "coarse_family", "average_voltage",
    "capacity_grav", "energy_grav", "max_delta_volume", "stability_worst", "final_triage_score",
]
missing_candidate_cols = [c for c in required_candidate_cols if c not in candidates.columns]
if missing_candidate_cols:
    raise ValueError(f"Candidate table missing required columns: {missing_candidate_cols}")

required_lit_cols = [
    "manual_candidate_id", "n_checked_rows", "best_annotation_status", "best_analogue_level",
    "best_relevance_score_0_to_3", "best_citation_title", "safe_interpretation",
]
missing_lit_cols = [c for c in required_lit_cols if c not in lit_summary.columns]
if missing_lit_cols:
    raise ValueError(f"Literature summary missing required columns: {missing_lit_cols}")

# Check that Notebook 09 really used the fully filled 252-row file.
checked_mask = lit_detail.get("manual_checked", pd.Series([], dtype=object)).astype(str).str.lower().isin(["true", "1", "yes", "y"])
n_checked_rows = int(checked_mask.sum())
n_checked_candidates = int(lit_detail.loc[checked_mask, "manual_candidate_id"].nunique()) if "manual_candidate_id" in lit_detail.columns else 0

if n_checked_candidates < 30:
    raise ValueError(
        "Notebook 09 detail file does not look complete. "
        f"Checked candidates = {n_checked_candidates}, checked rows = {n_checked_rows}. "
        "Expected 30 candidates with manual annotations before Notebook 10."
    )

print("Loaded candidate table:", TOP30_PATH, candidates.shape)
print("Loaded literature summary:", LIT_SUMMARY_PATH, lit_summary.shape)
print("Loaded literature detail:", LIT_DETAIL_PATH, lit_detail.shape)
print("Notebook 09 checked rows:", n_checked_rows)
print("Notebook 09 checked candidates:", n_checked_candidates)

# Merge candidate triage and literature evidence.
merge_keys = ["manual_candidate_id"]
work = candidates.merge(
    lit_summary.drop(columns=[c for c in ["shortlist_priority_rank", "battery_formula", "formula_discharge", "framework_formula", "coarse_family"] if c in lit_summary.columns], errors="ignore"),
    on=merge_keys,
    how="left",
    validate="one_to_one",
)

# Numeric cleanup.
for col in [
    "shortlist_priority_rank", "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume",
    "stability_worst", "final_triage_score", "top10_probability_mc", "top20_probability_mc",
    "rank_iqr_mc", "uncertainty_penalty_norm", "ad_penalty_norm", "best_relevance_score_0_to_3", "n_checked_rows",
]:
    if col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce")

print("Merged working table:", work.shape)
display(work[["manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_charge", "formula_discharge", "final_triage_score", "best_annotation_status", "best_relevance_score_0_to_3", "n_checked_rows"]].head(30))

log_event("load_inputs", "INFO", "Loaded Notebook 08/13 inputs", {
    "top30_path": str(TOP30_PATH),
    "lit_summary_path": str(LIT_SUMMARY_PATH),
    "lit_detail_path": str(LIT_DETAIL_PATH),
    "checked_rows": n_checked_rows,
    "checked_candidates": n_checked_candidates,
})
save_event_log()


In [ ]:

# ============================================================
# Cell 3 — Scoring helpers for DFT candidate selection
# ============================================================

ELEMENT_RE = re.compile(r"[A-Z][a-z]?")

DFT_U_ELEMENTS = {"Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"}
HEAVY_RISK_ELEMENTS = {"As", "Sb", "Bi", "Pb", "Hg", "Cd"}
HALIDE_ELEMENTS = {"F", "Cl", "Br", "I"}
COMMON_LIGHT_ELEMENTS = {"Na", "Li", "K", "O", "F", "P", "S", "C", "H", "B", "N", "Si"}

POSITIVE_LIT_STATUSES = {
    "exact_or_near_exact_formula",
    "same_framework_analogue",
    "same_family_same_transition_metal",
    "same_family_analogue",
}
WEAK_LIT_STATUSES = {"general_background_only"}
NEGATIVE_LIT_STATUSES = {"no_relevant_literature_found"}


def formula_elements(*formula_parts) -> set[str]:
    txt = "".join(clean_str(x) for x in formula_parts)
    return set(ELEMENT_RE.findall(txt))


def minmax_norm(s: pd.Series, higher_is_better=True) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn, mx = float(x.min()), float(x.max())
    if abs(mx - mn) < 1e-12:
        out = pd.Series(np.ones(len(s)) * 0.5, index=s.index)
    else:
        out = (x - mn) / (mx - mn)
    if not higher_is_better:
        out = 1 - out
    return out.fillna(0.0).clip(0, 1)


def lit_support_score(row) -> float:
    status = clean_str(row.get("best_annotation_status"))
    level = clean_str(row.get("best_analogue_level"))
    rel = row.get("best_relevance_score_0_to_3")
    try:
        rel = float(rel)
    except Exception:
        rel = 0.0
    # Conservative mapping: exact experimental/computed formula support is useful but not validation.
    if status == "exact_or_near_exact_formula":
        base = 0.75
    elif status == "same_framework_analogue":
        base = 0.55
    elif status == "same_family_same_transition_metal":
        base = 0.35
    elif status == "same_family_analogue":
        base = 0.25
    elif status == "general_background_only":
        base = 0.08
    elif status == "no_relevant_literature_found":
        base = 0.0
    else:
        base = 0.0
    if level == "exact_formula":
        base += 0.15
    elif level == "near_exact_formula":
        base += 0.10
    elif level == "same_framework":
        base += 0.05
    return float(np.clip(base + 0.05 * rel, 0.0, 1.0))


def dft_feasibility_score(row) -> float:
    elems = formula_elements(row.get("formula_charge"), row.get("formula_discharge"), row.get("framework_formula"))
    score = 1.0
    n_elems = len(elems)
    if n_elems >= 6:
        score -= 0.15
    if elems & HEAVY_RISK_ELEMENTS:
        score -= 0.30
    if "As" in elems:
        score -= 0.30
    if "H" in elems:
        score -= 0.20
    if len(elems & DFT_U_ELEMENTS) >= 2:
        score -= 0.08
    if "F" in elems and clean_str(row.get("coarse_family")).startswith("halide"):
        score -= 0.07
    try:
        dv = float(row.get("max_delta_volume"))
        if dv > 0.20:
            score -= 0.25
        elif dv > 0.15:
            score -= 0.10
    except Exception:
        pass
    try:
        st = float(row.get("stability_worst"))
        if st > 0.10:
            score -= 0.20
        elif st > 0.07:
            score -= 0.08
    except Exception:
        pass
    return float(np.clip(score, 0.0, 1.0))


def make_risk_tags(row) -> list[str]:
    elems = formula_elements(row.get("formula_charge"), row.get("formula_discharge"), row.get("framework_formula"))
    tags = []
    status = clean_str(row.get("best_annotation_status"))
    if status in NEGATIVE_LIT_STATUSES:
        tags.append("no_relevant_literature")
    if status in WEAK_LIT_STATUSES:
        tags.append("background_only_literature")
    if elems & HEAVY_RISK_ELEMENTS:
        tags.append("heavy_or_toxic_element:" + ",".join(sorted(elems & HEAVY_RISK_ELEMENTS)))
    if "As" in elems:
        tags.append("arsenate_or_As_risk")
    if "H" in elems:
        tags.append("H_or_hydration/proton_ambiguity")
    if "F" in elems and clean_str(row.get("coarse_family")).startswith("halide"):
        tags.append("fluoride_conversion_or_DFT+U_sensitivity")
    try:
        if float(row.get("max_delta_volume")) > MAX_VOLUME_CHANGE_FOR_PRIMARY:
            tags.append("large_volume_change")
    except Exception:
        pass
    try:
        if float(row.get("stability_worst")) > MAX_STABILITY_WORST_FOR_PRIMARY:
            tags.append("stability_near_threshold")
    except Exception:
        pass
    if str(row.get("out_of_domain_flag", "False")).lower() in ["true", "1", "yes"]:
        tags.append("out_of_domain")
    if str(row.get("high_uncertainty_flag", "False")).lower() in ["true", "1", "yes"]:
        tags.append("high_uncertainty")
    return tags


def hard_exclusion_reason(row) -> str:
    elems = formula_elements(row.get("formula_charge"), row.get("formula_discharge"), row.get("framework_formula"))
    reasons = []
    status = clean_str(row.get("best_annotation_status"))
    if HARD_EXCLUDE_NO_RELEVANT_LITERATURE and status in NEGATIVE_LIT_STATUSES:
        reasons.append("no relevant literature found")
    if HARD_EXCLUDE_AS_CONTAINING and "As" in elems:
        reasons.append("As-containing chemistry is too risky for primary DFT spot-check")
    if HARD_EXCLUDE_H_CONTAINING and "H" in elems:
        reasons.append("H-containing/protonated framework is ambiguous")
    if HARD_EXCLUDE_OUT_OF_DOMAIN and str(row.get("out_of_domain_flag", "False")).lower() in ["true", "1", "yes"]:
        reasons.append("out of applicability domain")
    if HARD_EXCLUDE_HIGH_UNCERTAINTY and str(row.get("high_uncertainty_flag", "False")).lower() in ["true", "1", "yes"]:
        reasons.append("high uncertainty flag")
    if HARD_EXCLUDE_SIMPLE_HALIDE_FLUORIDE_PRIMARY and clean_str(row.get("coarse_family")).startswith("halide"):
        reasons.append("simple halide/fluoride chemistry reserved, not primary")
    try:
        if float(row.get("max_delta_volume")) > MAX_VOLUME_CHANGE_FOR_PRIMARY:
            reasons.append(f"max_delta_volume > {MAX_VOLUME_CHANGE_FOR_PRIMARY}")
    except Exception:
        pass
    try:
        if float(row.get("stability_worst")) > MAX_STABILITY_WORST_FOR_PRIMARY:
            reasons.append(f"stability_worst > {MAX_STABILITY_WORST_FOR_PRIMARY}")
    except Exception:
        pass
    return "; ".join(reasons)


def selection_reason(row) -> str:
    return (
        f"rank={row.get('shortlist_priority_rank')}, triage={row.get('final_triage_score'):.3f}, "
        f"lit={row.get('best_annotation_status')}:{row.get('best_analogue_level')}, "
        f"voltage={row.get('average_voltage'):.2f} V, capacity={row.get('capacity_grav'):.1f} mAh/g, "
        f"ΔV={row.get('max_delta_volume'):.3f}, stability={row.get('stability_worst'):.3f}. "
        "Selected as DFT spot-check candidate, not as discovery claim."
    )


In [ ]:

# ============================================================
# Cell 4 — Score and classify candidates
# ============================================================

scored = work.copy()

# Normalize core ranking metrics.
scored["score_triage_norm"] = minmax_norm(scored["final_triage_score"], higher_is_better=True)
scored["score_rank_norm"] = 1 - minmax_norm(scored["shortlist_priority_rank"], higher_is_better=True)
scored["score_energy_norm"] = minmax_norm(scored["energy_grav"], higher_is_better=True)
scored["score_voltage_norm"] = minmax_norm(scored["average_voltage"], higher_is_better=True)
scored["score_stability_norm"] = minmax_norm(scored["stability_worst"], higher_is_better=False)
scored["score_volume_norm"] = minmax_norm(scored["max_delta_volume"], higher_is_better=False)
if "top10_probability_mc" in scored.columns:
    scored["score_rank_robust_norm"] = scored["top10_probability_mc"].fillna(0).clip(0, 1)
else:
    scored["score_rank_robust_norm"] = 0.0

scored["literature_support_score"] = scored.apply(lit_support_score, axis=1)
scored["dft_feasibility_score"] = scored.apply(dft_feasibility_score, axis=1)
scored["risk_tags"] = scored.apply(lambda r: "|".join(make_risk_tags(r)), axis=1)
scored["hard_exclusion_reason"] = scored.apply(hard_exclusion_reason, axis=1)
scored["primary_eligible"] = scored["hard_exclusion_reason"].astype(str).str.len() == 0

# Combined score: intentionally balances ML rank with DFT feasibility and literature support.
scored["dft_selection_score"] = (
    0.35 * scored["score_triage_norm"] +
    0.15 * scored["score_rank_robust_norm"] +
    0.15 * scored["literature_support_score"] +
    0.15 * scored["dft_feasibility_score"] +
    0.10 * scored["score_stability_norm"] +
    0.10 * scored["score_volume_norm"]
)

# Penalize weak-only literature but do not erase them from reserves.
scored.loc[scored["best_annotation_status"].isin(list(WEAK_LIT_STATUSES)), "dft_selection_score"] -= 0.10
scored.loc[scored["best_annotation_status"].isin(list(NEGATIVE_LIT_STATUSES)), "dft_selection_score"] -= 0.30
scored["dft_selection_score"] = scored["dft_selection_score"].clip(0, 1)

# Deduplicate exact battery-window duplicate families using formula_charge + formula_discharge.
scored["state_pair_key"] = (
    scored["formula_charge"].astype(str).str.replace(" ", "", regex=False) + "_" +
    scored["formula_discharge"].astype(str).str.replace(" ", "", regex=False)
)
scored["duplicate_state_pair_rank"] = scored.groupby("state_pair_key")["dft_selection_score"].rank(method="first", ascending=False)
scored["is_duplicate_lower_rank"] = scored["duplicate_state_pair_rank"] > 1
scored.loc[scored["is_duplicate_lower_rank"], "primary_eligible"] = False
scored.loc[scored["is_duplicate_lower_rank"], "hard_exclusion_reason"] = scored.loc[scored["is_duplicate_lower_rank"], "hard_exclusion_reason"].apply(lambda x: (clean_str(x) + "; duplicate state pair").strip("; "))

score_path = PROCESSED_DIR / "10_dft_candidate_selection_scores.csv"
scored.to_csv(score_path, index=False)

summary_cols = [
    "manual_candidate_id", "shortlist_priority_rank", "battery_formula", "formula_charge", "formula_discharge",
    "framework_formula", "coarse_family", "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume",
    "stability_worst", "final_triage_score", "top10_probability_mc", "best_annotation_status",
    "best_analogue_level", "best_relevance_score_0_to_3", "literature_support_score", "dft_feasibility_score",
    "dft_selection_score", "primary_eligible", "hard_exclusion_reason", "risk_tags",
]
print("Candidate scoring saved:", score_path)
display(scored[summary_cols].sort_values("dft_selection_score", ascending=False).head(30))


In [ ]:

# ============================================================
# Cell 5 — Select primary DFT candidates and optional control
# ============================================================

if PRIMARY_DFT_CANDIDATE_COUNT < 3 or PRIMARY_DFT_CANDIDATE_COUNT > 5:
    raise ValueError("PRIMARY_DFT_CANDIDATE_COUNT should be between 3 and 5 for reviewer-safe spot checks.")

# Selection priority:
# 1) explicit manual override,
# 2) reviewer-safe default IDs for this sodium case-study stage,
# 3) automatic score/diversity fallback.
if MANUAL_PRIMARY_CANDIDATE_IDS:
    selected_primary = scored[scored["manual_candidate_id"].isin(MANUAL_PRIMARY_CANDIDATE_IDS)].copy()
    missing_ids = sorted(set(MANUAL_PRIMARY_CANDIDATE_IDS) - set(selected_primary["manual_candidate_id"]))
    if missing_ids:
        raise ValueError(f"Manual primary candidate IDs not found: {missing_ids}")
    selected_primary["selection_method"] = "manual_override"
elif USE_REVIEWER_SAFE_DEFAULT_PRIMARY_IDS:
    default_ids = [cid for cid in REVIEWER_SAFE_DEFAULT_PRIMARY_IDS if cid in set(scored["manual_candidate_id"])]
    selected_primary = scored[scored["manual_candidate_id"].isin(default_ids)].copy()
    selected_primary["_default_order"] = selected_primary["manual_candidate_id"].map({cid: i for i, cid in enumerate(default_ids)})
    selected_primary = selected_primary.sort_values("_default_order").drop(columns=["_default_order"])
    selected_primary["selection_method"] = "reviewer_safe_default_case_study_selection"
    if len(selected_primary) < 3:
        log_event("candidate_selection", "WARNING", "Reviewer-safe defaults produced fewer than 3 candidates; falling back to automatic selection.", {"default_ids_found": default_ids})
        USE_FALLBACK_AUTOMATIC = True
    else:
        USE_FALLBACK_AUTOMATIC = False
else:
    USE_FALLBACK_AUTOMATIC = True

if 'USE_FALLBACK_AUTOMATIC' in globals() and USE_FALLBACK_AUTOMATIC:
    # Automatic selection with diversity pressure.
    eligible = scored[scored["primary_eligible"]].copy()
    eligible = eligible.sort_values("dft_selection_score", ascending=False).reset_index(drop=True)

    selected_rows = []
    used_chem_systems = set()
    used_formula_charges = set()

    # Pass 1: select diverse high-score candidates, max one per exact charged formula and chemical system if possible.
    for _, r in eligible.iterrows():
        if len(selected_rows) >= PRIMARY_DFT_CANDIDATE_COUNT:
            break
        chem = clean_str(r.get("chemical_system_uid"))
        charge_formula = clean_str(r.get("formula_charge"))
        # Avoid overloading one chemistry if enough alternatives remain.
        if chem in used_chem_systems and len(selected_rows) < max(2, PRIMARY_DFT_CANDIDATE_COUNT - 1):
            continue
        if charge_formula in used_formula_charges:
            continue
        selected_rows.append(r)
        used_chem_systems.add(chem)
        used_formula_charges.add(charge_formula)

    # Pass 2: fill remaining slots by score if diversity pressure was too strict.
    if len(selected_rows) < PRIMARY_DFT_CANDIDATE_COUNT:
        selected_ids = {r["manual_candidate_id"] for r in selected_rows}
        for _, r in eligible.iterrows():
            if len(selected_rows) >= PRIMARY_DFT_CANDIDATE_COUNT:
                break
            if r["manual_candidate_id"] in selected_ids:
                continue
            selected_rows.append(r)
            selected_ids.add(r["manual_candidate_id"])

    selected_primary = pd.DataFrame(selected_rows).copy() if selected_rows else pd.DataFrame(columns=scored.columns)
    selected_primary["selection_method"] = "automatic_score_with_diversity"

selected_primary["selection_role"] = "primary_dft_spotcheck"
selected_primary["selection_reason"] = selected_primary.apply(selection_reason, axis=1) if len(selected_primary) else ""

# Optional control: highest-ranked exact experimental/simple oxide not already selected.
selected_controls = pd.DataFrame(columns=scored.columns)
if INCLUDE_OPTIONAL_CONTROL:
    if MANUAL_CONTROL_CANDIDATE_IDS:
        selected_controls = scored[scored["manual_candidate_id"].isin(MANUAL_CONTROL_CANDIDATE_IDS)].copy()
        selected_controls["selection_method"] = "manual_control_override"
    elif USE_REVIEWER_SAFE_DEFAULT_PRIMARY_IDS and REVIEWER_SAFE_DEFAULT_CONTROL_IDS:
        selected_ids = set(selected_primary["manual_candidate_id"])
        default_control_ids = [cid for cid in REVIEWER_SAFE_DEFAULT_CONTROL_IDS if cid in set(scored["manual_candidate_id"]) and cid not in selected_ids]
        selected_controls = scored[scored["manual_candidate_id"].isin(default_control_ids)].copy().head(MAX_OPTIONAL_CONTROLS)
        selected_controls["selection_method"] = "reviewer_safe_default_control"
    else:
        selected_ids = set(selected_primary["manual_candidate_id"])
        control_pool = scored[
            (~scored["manual_candidate_id"].isin(selected_ids)) &
            (scored["best_annotation_status"] == "exact_or_near_exact_formula") &
            (scored["coarse_family"].astype(str).str.contains("oxide", case=False, na=False)) &
            (scored["primary_eligible"] | scored["hard_exclusion_reason"].astype(str).eq(""))
        ].copy()
        control_pool = control_pool.sort_values(["shortlist_priority_rank", "dft_selection_score"], ascending=[True, False])
        selected_controls = control_pool.head(MAX_OPTIONAL_CONTROLS).copy()
        selected_controls["selection_method"] = "automatic_optional_control"
    if len(selected_controls):
        selected_controls["selection_role"] = "optional_control_not_primary"
        selected_controls["selection_reason"] = selected_controls.apply(
            lambda r: "Optional control/calibration candidate with strong literature and simple oxide chemistry; not counted as a primary DFT spot-check.",
            axis=1,
        )

selected_ids = set(selected_primary["manual_candidate_id"])
control_ids = set(selected_controls["manual_candidate_id"]) if len(selected_controls) else set()
reserve = scored[~scored["manual_candidate_id"].isin(selected_ids | control_ids)].copy()
reserve = reserve.sort_values("dft_selection_score", ascending=False).reset_index(drop=True)

primary_path = PROCESSED_DIR / "10_dft_primary_selection.csv"
control_path = PROCESSED_DIR / "10_dft_optional_control_selection.csv"
reserve_path = PROCESSED_DIR / "10_dft_reserve_candidates.csv"

selected_primary.to_csv(primary_path, index=False)
selected_controls.to_csv(control_path, index=False)
reserve.to_csv(reserve_path, index=False)

print("Primary DFT candidates:", len(selected_primary))
display(selected_primary[summary_cols + ["selection_role", "selection_reason"]])

if len(selected_controls):
    print("Optional control candidate(s):", len(selected_controls))
    display(selected_controls[summary_cols + ["selection_role", "selection_reason"]])
else:
    print("No optional control candidate selected.")

print("Reserve candidates saved:", reserve_path)



## Structure retrieval logic

Notebook 10 can generate complete DFT input folders only if it can resolve structures for both charged and discharged states.

Priority order:

1. Use material IDs if found in earlier project files.
2. Search Materials Project by exact formula if material IDs are absent.
3. If no structure is resolved, save a clear TODO row for manual structure resolution.

The notebook will never print or save your Materials Project API key.


In [ ]:

# ============================================================
# Cell 6 — Optional discovery of material IDs from earlier outputs
# ============================================================

SEARCH_FOR_MP_IDS = True
PROJECT_ROOT = Path("outputs")

MP_ID_PATTERN = re.compile(r"mp-\d+")
POSSIBLE_ID_COL_HINTS = [
    "material_id", "material_ids", "charged_material_id", "discharged_material_id", "framework_material_id",
    "charged_id", "discharged_id", "mpid", "mp_id", "task_id", "ids",
]
POSSIBLE_MATCH_COLS = ["record_index", "electrode_uid", "battery_formula", "formula_charge", "formula_discharge", "framework_formula", "manual_candidate_id"]


def find_mp_ids_in_value(x) -> list[str]:
    return sorted(set(MP_ID_PATTERN.findall(clean_str(x))))


def discover_candidate_material_ids(selected_table: pd.DataFrame) -> pd.DataFrame:
    if not SEARCH_FOR_MP_IDS or not PROJECT_ROOT.exists():
        return pd.DataFrame()

    csv_files = sorted(PROJECT_ROOT.rglob("*.csv"))
    rows = []
    selected = selected_table.copy()

    for path in csv_files:
        # Avoid searching Notebook 10 outputs while rerunning.
        if "Notebook 10" in str(path):
            continue
        try:
            df = pd.read_csv(path, low_memory=False, nrows=None)
        except Exception:
            continue
        if df.empty:
            continue

        id_cols = [c for c in df.columns if any(h.lower() in c.lower() for h in POSSIBLE_ID_COL_HINTS)]
        if not id_cols:
            # Sometimes MP IDs are embedded in any object-like column; keep scan limited to avoid slowness.
            id_cols = [c for c in df.columns if df[c].dtype == "object"][:20]

        match_cols = [c for c in POSSIBLE_MATCH_COLS if c in df.columns and c in selected.columns]
        if not match_cols:
            continue

        for _, cand in selected.iterrows():
            mask = pd.Series([False] * len(df))
            for c in match_cols:
                val = clean_str(cand.get(c))
                if val:
                    mask = mask | (df[c].astype(str).str.strip() == val)
            hits = df[mask]
            if hits.empty:
                continue
            for _, hit in hits.head(20).iterrows():
                ids = []
                for c in id_cols:
                    ids.extend(find_mp_ids_in_value(hit.get(c)))
                ids = sorted(set(ids))
                if ids:
                    rows.append({
                        "manual_candidate_id": cand.get("manual_candidate_id"),
                        "source_file": str(path),
                        "matched_columns": "|".join(match_cols),
                        "mp_ids_found": "|".join(ids),
                    })

    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=["manual_candidate_id", "source_file", "matched_columns", "mp_ids_found"])

selection_for_structures = pd.concat([selected_primary, selected_controls], ignore_index=True, sort=False)
mp_id_discovery = discover_candidate_material_ids(selection_for_structures)
mp_id_discovery_path = AUDIT_DIR / "10_mp_id_discovery_audit.csv"
mp_id_discovery.to_csv(mp_id_discovery_path, index=False)

print("MP ID discovery rows:", len(mp_id_discovery))
if len(mp_id_discovery):
    display(mp_id_discovery.head(50))
else:
    print("No MP IDs discovered from earlier CSV files. Notebook will try formula-based MP search if API is available.")


In [ ]:

# ============================================================
# Cell 7 — Materials Project structure resolution helpers
# : materials project API key prompt + robust formula/chemsys search
# ============================================================

try:
    from mp_api.client import MPRester
    MP_API_AVAILABLE = True
except Exception as e:
    MPRester = None
    MP_API_AVAILABLE = False
    log_event("mp_api", "WARNING", "mp-api is not importable", {"error": str(e)})

try:
    from pymatgen.core import Structure, Composition
    PYMATGEN_AVAILABLE = True
except Exception as e:
    Structure = Composition = None
    PYMATGEN_AVAILABLE = False
    log_event("pymatgen", "WARNING", "pymatgen is not importable", {"error": str(e)})

print("mp-api available:", MP_API_AVAILABLE)
print("pymatgen available:", PYMATGEN_AVAILABLE)


def get_mp_api_key() -> str | None:
    """Return MP API key without printing/saving it."""
    key = os.environ.get("MP_API_KEY", "").strip()
    if key:
        print("MP_API_KEY found in environment. Key is not printed or saved.")
        return key
    if ASK_FOR_MP_API_KEY_IF_MISSING:
        key = getpass("Enter Materials Project API key (hidden; not saved): ").strip()
        if key:
            print("MP API key received from hidden prompt. Key is not printed or saved.")
        return key or None
    return None


def open_mpr():
    if not MP_API_AVAILABLE:
        return None
    key = get_mp_api_key()
    if not key:
        log_event("mp_api", "WARNING", "No MP_API_KEY found; skipping MP structure retrieval")
        return None
    try:
        return MPRester(key)
    except Exception as e:
        log_event("mp_api", "ERROR", "Could not initialize MPRester", {"error": str(e)})
        return None


def get_doc_value(doc, field: str, default=None):
    if doc is None:
        return default
    if isinstance(doc, dict):
        return doc.get(field, default)
    return getattr(doc, field, default)


def material_id_to_string(mid) -> str:
    return clean_str(mid).replace("MaterialsProject", "")


def parse_composition_safe(formula_or_comp):
    if Composition is None or formula_or_comp is None:
        return None
    try:
        if isinstance(formula_or_comp, Composition):
            return formula_or_comp
        return Composition(str(formula_or_comp))
    except Exception:
        return None


def composition_key(formula_or_comp) -> tuple | None:
    comp = parse_composition_safe(formula_or_comp)
    if comp is None:
        return None
    try:
        red = comp.reduced_composition
        return tuple(sorted((el.symbol, round(float(amount), 8)) for el, amount in red.items()))
    except Exception:
        try:
            frac = comp.fractional_composition
            return tuple(sorted((el.symbol, round(float(amount), 8)) for el, amount in frac.items()))
        except Exception:
            return None


def doc_composition_key(doc) -> tuple | None:
    for field in ["composition", "formula_pretty"]:
        val = get_doc_value(doc, field, None)
        key = composition_key(val)
        if key is not None:
            return key
    st = get_doc_value(doc, "structure", None)
    if st is not None:
        try:
            return composition_key(st.composition)
        except Exception:
            pass
    return None


def same_reduced_composition(a, b) -> bool:
    ka = composition_key(a)
    kb = composition_key(b)
    return (ka is not None) and (kb is not None) and (ka == kb)


def formula_search_variants(formula: str) -> list[str]:
    """Generate MP-friendly formula variants, including canonical Composition formulas.

    Literature-search grouped formulas like Na3CoPO4CO3 are useful for Scholar, but MP
    search generally expects plain elemental formulas. For condensed carbonophosphates,
    both Na3CoPCO7 and Na3CoCPO7 may appear depending on formula ordering.
    """
    f = clean_str(formula).replace(" ", "")
    variants = []
    if f:
        variants.append(f)
    comp = parse_composition_safe(f)
    if comp is not None:
        for attr in ["reduced_formula", "alphabetical_formula", "formula"]:
            try:
                variants.append(clean_str(getattr(comp, attr)).replace(" ", ""))
            except Exception:
                pass
        try:
            variants.append(clean_str(comp.reduced_composition.formula).replace(" ", ""))
            variants.append(clean_str(comp.reduced_composition.alphabetical_formula).replace(" ", ""))
        except Exception:
            pass
    # Common condensed polyoxo re-ordering variants: PCO7 <-> CPO7, AsCO7 <-> CAsO7, SiCO7 <-> CSiO7.
    extra = []
    for v in variants:
        extra.append(v.replace("PCO7", "CPO7"))
        extra.append(v.replace("AsCO7", "CAsO7"))
        extra.append(v.replace("SiCO7", "CSiO7"))
        extra.append(v.replace("CSO7", "CSO7"))
    variants.extend(extra)
    out = []
    seen = set()
    for v in variants:
        v = clean_str(v).replace(" ", "")
        if v and v not in seen:
            out.append(v)
            seen.add(v)
    return out


def chemsys_from_formula(formula: str) -> str:
    comp = parse_composition_safe(formula)
    if comp is None:
        # fallback by regex, less reliable
        elems = sorted(set(ELEMENT_RE.findall(clean_str(formula))))
        return "-".join(elems)
    try:
        return comp.chemical_system
    except Exception:
        elems = sorted(el.symbol for el in comp.elements)
        return "-".join(elems)


def _summary_search(mpr, **kwargs):
    """Call summary search with robust field fallbacks."""
    fields_options = [
        ["material_id", "formula_pretty", "energy_above_hull", "symmetry", "structure", "composition"],
        ["material_id", "formula_pretty", "energy_above_hull", "structure", "composition"],
        ["material_id", "formula_pretty", "energy_above_hull", "composition"],
        ["material_id", "formula_pretty", "structure"],
        None,
    ]
    last_error = None
    for fields in fields_options:
        try:
            if fields is None:
                return list(mpr.materials.summary.search(**kwargs) or [])
            return list(mpr.materials.summary.search(**kwargs, fields=fields) or [])
        except Exception as e:
            last_error = e
            continue
    if last_error:
        log_event("mp_api", "WARNING", "Summary search failed for kwargs", {"kwargs": kwargs, "error": str(last_error)})
    return []


def summary_search_by_formula(mpr, formula: str):
    """Robust formula search across possible mp-api versions.

    Strategy:
    1) Try exact formula variants.
    2) Try chemical-system search and filter to the same reduced composition.
    """
    formula = clean_str(formula)
    if not formula or mpr is None:
        return []

    all_docs = []
    # Exact formula variants first.
    for variant in formula_search_variants(formula):
        for kwargs in [
            {"formula": variant},
            {"formula": [variant]},
            {"formula_pretty": variant},
            {"chemsys_formula": variant},
        ]:
            docs = _summary_search(mpr, **kwargs)
            if docs:
                all_docs.extend(docs)

    # Chemical-system fallback: often succeeds when formula ordering differs.
    chemsys = chemsys_from_formula(formula)
    if chemsys:
        for kwargs in [{"chemsys": chemsys}, {"chemsys": [chemsys]}]:
            docs = _summary_search(mpr, **kwargs)
            if docs:
                # Keep exact reduced-composition matches first; if no comp key, keep as fallback.
                exact = [d for d in docs if doc_composition_key(d) == composition_key(formula)]
                all_docs.extend(exact)

    # Deduplicate by material_id.
    dedup = []
    seen = set()
    for d in all_docs:
        mid = material_id_to_string(get_doc_value(d, "material_id", ""))
        key = mid or clean_str(get_doc_value(d, "formula_pretty", ""))
        if key and key not in seen:
            dedup.append(d)
            seen.add(key)
    return dedup


def choose_best_summary_doc(docs: list, target_formula: str):
    if not docs:
        return None
    target_key = composition_key(target_formula)
    scored_docs = []
    target = clean_str(target_formula).replace(" ", "")
    for doc in docs:
        mid = material_id_to_string(get_doc_value(doc, "material_id", ""))
        pretty = clean_str(get_doc_value(doc, "formula_pretty", "")).replace(" ", "")
        eah = get_doc_value(doc, "energy_above_hull", np.nan)
        try:
            eah_num = float(eah)
        except Exception:
            eah_num = np.inf
        dkey = doc_composition_key(doc)
        same_comp_penalty = 0 if (target_key is not None and dkey == target_key) else 10
        exact_formula_penalty = 0 if pretty == target else 1
        has_structure_penalty = 0 if get_doc_value(doc, "structure", None) is not None else 1
        scored_docs.append((same_comp_penalty, exact_formula_penalty, has_structure_penalty, eah_num, mid, doc))
    scored_docs.sort(key=lambda x: (x[0], x[1], x[2], x[3], x[4]))
    best = scored_docs[0][5]
    # If no same-composition doc exists, treat as not found rather than accepting a wrong chemical system hit.
    if scored_docs[0][0] >= 10:
        return None
    return best


def get_structure_for_material_id(mpr, material_id: str):
    material_id = clean_str(material_id)
    if not material_id or mpr is None:
        return None
    # Try several mp-api versions.
    for fn in [
        lambda: mpr.get_structure_by_material_id(material_id),
        lambda: mpr.materials.get_structure_by_material_id(material_id),
    ]:
        try:
            st = fn()
            if st is not None:
                return st
        except Exception:
            pass
    try:
        docs = _summary_search(mpr, material_ids=[material_id])
        if docs:
            return get_doc_value(docs[0], "structure")
    except Exception:
        pass
    return None


def resolve_structure_by_formula(mpr, formula: str):
    docs = summary_search_by_formula(mpr, formula)
    doc = choose_best_summary_doc(docs, formula)
    if doc is None:
        return {
            "formula": formula,
            "material_id": "",
            "formula_pretty": "",
            "energy_above_hull": np.nan,
            "structure": None,
            "resolution_status": "not_found",
        }
    mid = material_id_to_string(get_doc_value(doc, "material_id", ""))
    structure = get_doc_value(doc, "structure", None)
    if structure is None and mid:
        structure = get_structure_for_material_id(mpr, mid)
    return {
        "formula": formula,
        "material_id": mid,
        "formula_pretty": clean_str(get_doc_value(doc, "formula_pretty", "")),
        "energy_above_hull": get_doc_value(doc, "energy_above_hull", np.nan),
        "structure": structure,
        "resolution_status": "resolved" if structure is not None else "id_found_structure_missing",
    }


In [ ]:

# ============================================================
# Cell 8 — Resolve charged/discharged structures for selected candidates
# ============================================================

mpr = open_mpr() if RUN_MP_STRUCTURE_RESOLUTION else None

resolution_rows = []
resolved_structures = {}  # key: (candidate_id, state) -> Structure-like object

for _, cand in selection_for_structures.iterrows():
    cid = clean_str(cand.get("manual_candidate_id"))
    for state_name, formula_col in [("charged", "formula_charge"), ("discharged", "formula_discharge")]:
        formula = clean_str(cand.get(formula_col))
        result = {
            "manual_candidate_id": cid,
            "selection_role": clean_str(cand.get("selection_role")),
            "state": state_name,
            "target_formula": formula,
            "resolved_material_id": "",
            "resolved_formula_pretty": "",
            "resolved_energy_above_hull": np.nan,
            "resolution_status": "not_attempted",
            "resolution_note": "",
        }

        if mpr is None:
            result["resolution_status"] = "mp_api_not_available_or_key_missing"
            result["resolution_note"] = "Set MP_API_KEY or use the hidden getpass prompt, then rerun this cell; alternatively manually place structures into dft_inputs."
        else:
            try:
                res = resolve_structure_by_formula(mpr, formula)
                result["resolved_material_id"] = res.get("material_id", "")
                result["resolved_formula_pretty"] = res.get("formula_pretty", "")
                result["resolved_energy_above_hull"] = res.get("energy_above_hull", np.nan)
                result["resolution_status"] = res.get("resolution_status", "unknown")
                if res.get("structure") is not None:
                    resolved_structures[(cid, state_name)] = res["structure"]
                else:
                    result["resolution_note"] = "Formula search did not return a usable structure. Manual MP ID may be required."
            except Exception as e:
                result["resolution_status"] = "error"
                result["resolution_note"] = str(e)
                log_event("structure_resolution", "WARNING", "Structure resolution failed", {"candidate": cid, "state": state_name, "formula": formula, "error": str(e)})

        resolution_rows.append(result)

structure_resolution = pd.DataFrame(resolution_rows)
structure_resolution_path = PROCESSED_DIR / "10_dft_structure_resolution_table.csv"
structure_resolution.to_csv(structure_resolution_path, index=False)

print("Structure resolution rows:", len(structure_resolution))
print("Resolved structures in memory:", len(resolved_structures))
display(structure_resolution)
save_event_log()


In [ ]:

# ============================================================
# Cell 9 — DFT input writer helpers
# ============================================================

# Pymatgen input generators are optional.
try:
    from pymatgen.io.vasp.sets import MPRelaxSet
    from pymatgen.io.vasp.inputs import Poscar, Kpoints, Incar
    PYMATGEN_VASP_AVAILABLE = True
except Exception as e:
    MPRelaxSet = Poscar = Kpoints = Incar = None
    PYMATGEN_VASP_AVAILABLE = False
    log_event("vasp_input", "WARNING", "pymatgen VASP input generators unavailable", {"error": str(e)})


def structure_to_formula(structure) -> str:
    try:
        return structure.composition.reduced_formula
    except Exception:
        return "unknown_formula"


def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def write_structure_files(structure, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    wrote = []
    try:
        structure.to(filename=str(out_dir / "structure.cif"))
        wrote.append("structure.cif")
    except Exception as e:
        log_event("write_structure", "WARNING", "Could not write CIF", {"out_dir": str(out_dir), "error": str(e)})
    try:
        structure.to(fmt="poscar", filename=str(out_dir / "POSCAR"))
        wrote.append("POSCAR")
    except Exception as e:
        log_event("write_structure", "WARNING", "Could not write POSCAR", {"out_dir": str(out_dir), "error": str(e)})
    return wrote


def write_vasp_template_files(structure, out_dir: Path, candidate_row: pd.Series, state: str):
    """Write MPRelaxSet if possible; otherwise write conservative template INCAR/KPOINTS.
    No POTCAR file is written.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    files_written = []

    # Try MPRelaxSet first. potcar_spec=True avoids writing licensed POTCAR contents.
    if GENERATE_VASP_INPUTS and PYMATGEN_VASP_AVAILABLE and structure is not None:
        try:
            user_incar = {
                "ENCUT": 520,
                "EDIFF": 1e-5,
                "EDIFFG": -0.03,
                "ISMEAR": 0,
                "SIGMA": 0.05,
                "LASPH": True,
                "LREAL": False,
                "ISPIN": 2,
                "NSW": 120,
                "IBRION": 2,
            }
            vset = MPRelaxSet(structure, user_incar_settings=user_incar)
            try:
                vset.write_input(str(out_dir), potcar_spec=True)
            except TypeError:
                # Older pymatgen may not support potcar_spec.
                vset.write_input(str(out_dir), make_dir_if_not_present=True)
                potcar_path = out_dir / "POTCAR"
                if potcar_path.exists():
                    potcar_path.unlink()
                    write_text(out_dir / "POTCAR_NOT_INCLUDED.txt", "POTCAR was intentionally removed. Generate it locally using your licensed VASP pseudopotentials.\n")
            files_written.extend([p.name for p in out_dir.iterdir() if p.is_file()])
        except Exception as e:
            log_event("vasp_input", "WARNING", "MPRelaxSet failed; writing template VASP files", {"out_dir": str(out_dir), "error": str(e)})

    # Ensure minimal templates exist.
    if GENERATE_VASP_INPUTS:
        if not (out_dir / "INCAR").exists():
            incar_text = f"""# VASP relaxation template generated by Notebook 10
# Verify DFT+U, MAGMOM, POTCAR choices, and convergence before production.
SYSTEM = {candidate_row.get('manual_candidate_id')} {state} {candidate_row.get('battery_formula')}
ENCUT = 520
EDIFF = 1E-5
EDIFFG = -0.03
ISMEAR = 0
SIGMA = 0.05
ISPIN = 2
LASPH = .TRUE.
LREAL = .FALSE.
NSW = 120
IBRION = 2
ISIF = 3
PREC = Accurate
ADDGRID = .TRUE.
# Recommended next steps:
# 1. Set MAGMOM explicitly for transition metals.
# 2. Verify LDAU/LDAUU/LDAUJ against your chosen functional and MP compatibility target.
# 3. Run convergence tests for ENCUT and k-point density.
"""
            write_text(out_dir / "INCAR", incar_text)
            files_written.append("INCAR")
        if not (out_dir / "KPOINTS").exists():
            kpoints_text = """Automatic mesh
0
Gamma
4 4 4
0 0 0
"""
            write_text(out_dir / "KPOINTS", kpoints_text)
            files_written.append("KPOINTS")
        if not (out_dir / "POTCAR.spec").exists():
            elems = sorted(formula_elements(candidate_row.get("formula_charge"), candidate_row.get("formula_discharge")))
            write_text(out_dir / "POTCAR.spec", "# POTCAR is not included. Suggested element list only; verify PAW datasets locally.\n" + "\n".join(elems) + "\n")
            files_written.append("POTCAR.spec")

    return sorted(set(files_written))


def write_qe_placeholder(structure, out_dir: Path, candidate_row: pd.Series, state: str):
    if not GENERATE_QE_PLACEHOLDER_INPUTS:
        return []
    formula = structure_to_formula(structure) if structure is not None else clean_str(candidate_row.get("formula_discharge"))
    text = f"""! Quantum ESPRESSO placeholder generated by Notebook 10
! This is not production-ready. Fill pseudopotentials, ecutwfc/ecutrho, occupations, magnetism, and k-points.
! Candidate: {candidate_row.get('manual_candidate_id')} | State: {state} | Formula: {formula}
&CONTROL
  calculation = 'vc-relax',
  prefix = '{safe_filename(str(candidate_row.get('manual_candidate_id')) + '_' + state)}',
  outdir = './tmp',
  pseudo_dir = './pseudo'
/
&SYSTEM
  ibrav = 0,
  nat = REPLACE_ME,
  ntyp = REPLACE_ME,
  ecutwfc = 70,
  ecutrho = 560,
  occupations = 'smearing',
  smearing = 'gaussian',
  degauss = 0.01,
  nspin = 2
/
&ELECTRONS
  conv_thr = 1.0d-6
/
&IONS
/
&CELL
/
ATOMIC_SPECIES
! Fill species and pseudopotentials locally.
ATOMIC_POSITIONS crystal
! Use structure.cif/POSCAR to generate this section.
K_POINTS automatic
4 4 4 0 0 0
CELL_PARAMETERS angstrom
! Use structure.cif/POSCAR to generate this section.
"""
    write_text(out_dir / "qe_vc_relax_TEMPLATE.in", text)
    return ["qe_vc_relax_TEMPLATE.in"]


def write_run_script(out_dir: Path):
    script = """#!/bin/bash
#SBATCH -job-name=dft_spotcheck
#SBATCH -nodes=1
#SBATCH -ntasks=32
#SBATCH -time=24:00:00
#SBATCH -partition=compute
#SBATCH -output=slurm-%j.out
#SBATCH -error=slurm-%j.err

# Edit module/mpi commands for your cluster.
# module purge
# module load vasp

# VASP example:
# srun vasp_std > vasp.out

# Never commit POTCAR or licensed pseudopotential files to the public repository.
"""
    write_text(out_dir / "run_vasp_slurm_TEMPLATE.sh", script)
    return ["run_vasp_slurm_TEMPLATE.sh"]


In [ ]:

# ============================================================
# Cell 10 — Generate DFT input folders
# ============================================================

manifest_rows = []

for _, cand in selection_for_structures.iterrows():
    cid = clean_str(cand.get("manual_candidate_id"))
    role = clean_str(cand.get("selection_role"))
    folder_name = f"{cid}_{safe_filename(cand.get('battery_formula'))}_{safe_filename(cand.get('formula_discharge'))}"
    cand_dir = DFT_INPUT_DIR / folder_name
    cand_dir.mkdir(parents=True, exist_ok=True)

    # Candidate-level metadata.
    candidate_meta = {
        "manual_candidate_id": cid,
        "selection_role": role,
        "battery_formula": clean_str(cand.get("battery_formula")),
        "formula_charge": clean_str(cand.get("formula_charge")),
        "formula_discharge": clean_str(cand.get("formula_discharge")),
        "framework_formula": clean_str(cand.get("framework_formula")),
        "coarse_family": clean_str(cand.get("coarse_family")),
        "average_voltage": cand.get("average_voltage"),
        "capacity_grav": cand.get("capacity_grav"),
        "energy_grav": cand.get("energy_grav"),
        "max_delta_volume": cand.get("max_delta_volume"),
        "stability_worst": cand.get("stability_worst"),
        "final_triage_score": cand.get("final_triage_score"),
        "best_annotation_status": clean_str(cand.get("best_annotation_status")),
        "best_analogue_level": clean_str(cand.get("best_analogue_level")),
        "best_citation_title": clean_str(cand.get("best_citation_title")),
        "selection_reason": clean_str(cand.get("selection_reason")),
        "reviewer_safety_note": "DFT spot-check candidate only; do not claim discovery or experimental validation.",
    }
    write_json_safe(candidate_meta, cand_dir / "candidate_metadata.json")

    readme = f"""# {cid} — DFT spot-check input folder

Role: {role}
Battery window formula: {clean_str(cand.get('battery_formula'))}
Charged formula: {clean_str(cand.get('formula_charge'))}
Discharged formula: {clean_str(cand.get('formula_discharge'))}
Framework formula: {clean_str(cand.get('framework_formula'))}

Reviewer-safety note:
This folder supports an original DFT spot-check only. It does not establish experimental validation and must not be presented as a discovery claim.

Before production DFT:
1. Verify structures and MP IDs.
2. Verify magnetic initialization and DFT+U values.
3. Verify pseudopotentials and k-point/ENCUT convergence.
4. Do not commit POTCAR or licensed pseudopotential files.
"""
    write_text(cand_dir / "README_DFT_SPOTCHECK.md", readme)

    for state in ["charged", "discharged"]:
        formula = clean_str(cand.get("formula_charge" if state == "charged" else "formula_discharge"))
        state_dir = cand_dir / f"{state}_{safe_filename(formula)}"
        state_dir.mkdir(parents=True, exist_ok=True)
        structure = resolved_structures.get((cid, state))
        resolution_row = structure_resolution[(structure_resolution["manual_candidate_id"] == cid) & (structure_resolution["state"] == state)]
        resolution_info = resolution_row.iloc[0].to_dict() if len(resolution_row) else {}

        files_written = []
        if structure is not None:
            files_written.extend(write_structure_files(structure, state_dir))
            files_written.extend(write_vasp_template_files(structure, state_dir, cand, state))
            files_written.extend(write_qe_placeholder(structure, state_dir, cand, state))
            files_written.extend(write_run_script(state_dir))
            status = "input_files_written"
        else:
            status = "structure_missing_inputs_not_written"
            todo = f"""# Structure required for {cid} {state}

Target formula: {formula}
Resolution status: {resolution_info.get('resolution_status', 'not_attempted')}
Resolution note: {resolution_info.get('resolution_note', '')}

To continue:
1. Set MP_API_KEY and rerun Notebook 10, or
2. Manually choose an MP material ID for this formula, download the structure, and place POSCAR/structure.cif here.
"""
            write_text(state_dir / "STRUCTURE_REQUIRED.md", todo)
            files_written.append("STRUCTURE_REQUIRED.md")

        write_json_safe({
            "manual_candidate_id": cid,
            "state": state,
            "target_formula": formula,
            "generation_status": status,
            "resolution_info": resolution_info,
            "files_written": files_written,
        }, state_dir / "state_metadata.json")

        manifest_rows.append({
            "manual_candidate_id": cid,
            "selection_role": role,
            "state": state,
            "target_formula": formula,
            "state_dir": str(state_dir),
            "generation_status": status,
            "resolved_material_id": resolution_info.get("resolved_material_id", ""),
            "resolved_formula_pretty": resolution_info.get("resolved_formula_pretty", ""),
            "resolution_status": resolution_info.get("resolution_status", ""),
            "files_written": "|".join(files_written),
        })

manifest = pd.DataFrame(manifest_rows)
manifest_path = PROCESSED_DIR / "10_dft_input_generation_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("DFT input generation manifest saved:", manifest_path)
display(manifest)


In [ ]:

# ============================================================
# Cell 11 — Final decision and next-step package
# ============================================================



# Additional reviewer-safe structure-resolution diagnostics.
structure_resolution_status_counts = structure_resolution["resolution_status"].value_counts(dropna=False).to_dict() if len(structure_resolution) else {}
primary_resolution = structure_resolution[structure_resolution["selection_role"] == "primary_dft_spotcheck"].copy() if len(structure_resolution) else pd.DataFrame()
primary_resolved_states = int((primary_resolution.get("resolution_status", pd.Series(dtype=str)) == "resolved").sum()) if len(primary_resolution) else 0

primary_ids = selected_primary["manual_candidate_id"].tolist()
control_ids = selected_controls["manual_candidate_id"].tolist() if len(selected_controls) else []

primary_manifest = manifest[manifest["selection_role"] == "primary_dft_spotcheck"].copy() if len(manifest) else pd.DataFrame()
primary_states_expected = 2 * len(selected_primary)
primary_states_written = int((primary_manifest.get("generation_status", pd.Series(dtype=str)) == "input_files_written").sum()) if len(primary_manifest) else 0
primary_candidates_with_all_states = []
for cid, g in primary_manifest.groupby("manual_candidate_id") if len(primary_manifest) else []:
    if set(g["state"]) == {"charged", "discharged"} and (g["generation_status"] == "input_files_written").all():
        primary_candidates_with_all_states.append(cid)

if len(selected_primary) < 3:
    final_decision = "NO_GO_TOO_FEW_DFT_CANDIDATES"
elif primary_states_written >= 2 * min(3, len(selected_primary)):
    final_decision = "FULL_GO_RUN_DFT_SPOTCHECKS"
elif len(selected_primary) >= 3:
    final_decision = "CONDITIONAL_GO_STRUCTURE_RESOLUTION_REQUIRED"
else:
    final_decision = "NO_GO_REDESIGN_DFT_SELECTION"

final_info = {
    "notebook": "10_dft_candidate_selection_and_input_generation.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "final_decision": final_decision,
    "primary_candidate_count": int(len(selected_primary)),
    "primary_candidate_ids": primary_ids,
    "optional_control_ids": control_ids,
    "primary_states_expected": int(primary_states_expected),
    "primary_states_with_input_files_written": int(primary_states_written),
    "primary_states_resolved": int(primary_resolved_states),
    "structure_resolution_status_counts": structure_resolution_status_counts,
    "primary_candidates_with_both_states_written": primary_candidates_with_all_states,
    "mp_api_available": bool(MP_API_AVAILABLE),
    "pymatgen_available": bool(PYMATGEN_AVAILABLE),
    "pymatgen_vasp_available": bool(PYMATGEN_VASP_AVAILABLE),
    "structure_resolution_path": str(structure_resolution_path),
    "dft_manifest_path": str(manifest_path),
    "primary_selection_path": str(primary_path),
    "control_selection_path": str(control_path),
    "reserve_candidates_path": str(reserve_path),
    "dft_input_dir": str(DFT_INPUT_DIR),
    "next_notebook": "15_dft_results_parsing_and_validation.ipynb",
    "reviewer_safety_note": "Proceed only after structures, magnetic states, DFT+U, pseudopotentials, and convergence are verified."
}

write_json_safe(final_info, METADATA_DIR / "10_final_decision.json")

# Output manifest.
out_rows = []
for p in sorted(BASE_DIR.rglob("*")):
    if p.is_file():
        out_rows.append({
            "relative_path": str(p.relative_to(BASE_DIR)),
            "size_bytes": p.stat().st_size,
            "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        })
output_manifest = pd.DataFrame(out_rows)
output_manifest.to_csv(METADATA_DIR / "10_output_manifest.csv", index=False)

save_event_log()

print("FINAL DECISION:", final_decision)
print("Primary candidate IDs:", primary_ids)
print("Optional control IDs:", control_ids)
print("DFT input directory:", DFT_INPUT_DIR)
print("Primary states written:", primary_states_written, "/", primary_states_expected)

if final_decision == "CONDITIONAL_GO_STRUCTURE_RESOLUTION_REQUIRED":
    print("\nYou can proceed conceptually, but DFT input folders are incomplete until structures are resolved.")
    print("Set MP_API_KEY and rerun Notebook 10, or manually place POSCAR/CIF files in the generated state folders.")
elif final_decision == "FULL_GO_RUN_DFT_SPOTCHECKS":
    print("\nReady to run the generated DFT spot-check calculations after manual verification of input settings.")
else:
    print("\nDo not proceed to production DFT until the issue above is fixed.")

display(pd.DataFrame([final_info]).T.rename(columns={0: "value"}))



## What to do after Notebook 10

If the final decision is:

```text
FULL_GO_RUN_DFT_SPOTCHECKS
```

then manually inspect every generated DFT input folder before running VASP/QE.

If the final decision is:

```text
CONDITIONAL_GO_STRUCTURE_RESOLUTION_REQUIRED
```

then the candidate selection is ready, but structures were not automatically fetched. Set `MP_API_KEY`, install `mp-api`/`pymatgen`, and rerun the structure-resolution/input-generation cells.

Notebook 15 should only be started after you have completed at least 3 primary DFT spot-check calculations for both charged and discharged states, or after you explicitly decide to parse a smaller pilot subset as a dry run.
